In [ ]:
!pip install bertopic

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import torch
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
import umap
import hdbscan
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import json
import numpy as np
import os

In [ ]:
import re
import time
from tqdm import tqdm
from nltk.corpus import stopwords
import spacy
from spacy.lang.en.stop_words import STOP_WORDS
import glob
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter
import re
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk import word_tokenize
from string import punctuation
from nltk.corpus import stopwords
from nltk import word_tokenize
import nltk
from string import punctuation
punctuation += '$£&'
from nltk.corpus import stopwords
EN_STOPWORDS = set(stopwords.words('english'))

In [ ]:
import os
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Define the main project directory created after extracting the zip file
BASE_PROJECT_DIR = "./TAM_2526_Project/TAM 2526 Project"

# Dataset folder
DATA_DIR = os.path.join(BASE_PROJECT_DIR, "data")

# Output folders
BASE_OUTPUT_DIR = os.path.join(BASE_PROJECT_DIR, "outputs")
FIGURES_DIR = os.path.join(BASE_OUTPUT_DIR, "figures")
TABLES_DIR = os.path.join(BASE_OUTPUT_DIR, "tables")
MODELS_DIR = os.path.join(BASE_OUTPUT_DIR, "models")
EMBEDDINGS_DIR = os.path.join(BASE_OUTPUT_DIR, "embeddings")

# Create folders if they do not exist
for folder in [BASE_OUTPUT_DIR, FIGURES_DIR, TABLES_DIR, MODELS_DIR, EMBEDDINGS_DIR]:
    os.makedirs(folder, exist_ok=True)

# Helper functions
def get_figure_path(filename):
    return os.path.join(FIGURES_DIR, filename)

def get_table_path(filename):
    return os.path.join(TABLES_DIR, filename)

def get_model_path(filename):
    return os.path.join(MODELS_DIR, filename)

def get_embedding_path(filename):
    return os.path.join(EMBEDDINGS_DIR, filename)

## **DATASET**

The dataset consists of 35 transcripts of Donald Trump rally speeches delivered between 2019 and 2020.  
Each speech is stored as a `.txt` file, and the filename contains basic metadata about the event, including the location, month, day, and year of the rally.

The first step of the analysis is to load all text files into a structured `pandas` DataFrame.  
For each speech, the notebook stores the original filename and the full transcript. Then, additional metadata are extracted from the filename through regular expressions:

- `year`: the year of the rally, either 2019 or 2020;
- `month`: the abbreviated month of the rally;
- `location`: the place where the rally was held.

This metadata is necessary for the later stages of the project, especially for comparing topic distributions across time and geographical locations.  
After loading and metadata extraction, the speeches are sorted chronologically to preserve the temporal structure of the corpus.

In [ ]:
#importing the dataset

import zipfile
import os

zip_path = "TAM 2526 Project.zip"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("TAM_2526_Project")


os.listdir("TAM_2526_Project")


for root, dirs, files in os.walk("."):
    if "data" in dirs:
        print(os.path.join(root, "data"))

archive_path = DATA_DIR

In [ ]:
# Retrieve all .txt files recursively from the dataset folder.
#in case the archive path doessn't work, substitute it with the result of the previous cell

archive_path = "./TAM_2526_Project/TAM 2526 Project/data"
files = glob.glob(archive_path + '**/*.txt', recursive=True)


data = []
for file in files:
    filename = os.path.basename(file)
    with open(file, 'r', encoding='utf-8') as f:
        text = f.read()
    data.append({'filename': filename, 'text': text})


df = pd.DataFrame(data)

print(f"There are {len(df)} speeches in the dataset.")

# Extract metadata
# 1. Extract the year from the filename and store it as an integer.
df['year'] = df['filename'].str.extract(r'(2019|2020)').astype(int)

# 2. Extract the month abbreviation from the filename.
# A numerical month column is temporarily created to sort speeches chronologically.
month_pattern = r'(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)'
df['month'] = df['filename'].str.extract(month_pattern)

month_map = {'Jan':1, 'Feb':2, 'Mar':3, 'Apr':4, 'May':5, 'Jun':6,
             'Jul':7, 'Aug':8, 'Sep':9, 'Oct':10, 'Nov':11, 'Dec':12}

# Sort speeches by month, then remove the temporary numerical month column.
df['month_num'] = df['month'].map(month_map)
df = df.sort_values(by=['year', 'month_num']).drop(columns=['month_num']).reset_index(drop=True)


# 3. Extract the rally location by removing month, year, day numbers, and file extension from the filename.
df['location'] = df['filename'].str.replace(r'(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)', '', regex=True)
df['location'] = df['location'].str.replace(r'(2019|2020)', '', regex=True)

df['location'] = df['location'].str.replace(r'\\d{1,2}', '', regex=True)
df['location'] = df['location'].str.replace('.txt', '', regex=False)
df['location'] = df['location'].str.replace('_', ' ').str.strip().str.title()

df

In [ ]:
print(df['filename'][0])
print(df['text'][0])
print(df['month'][0])
print(df['year'][0])
print(df['location'][0])

## **INFORMATION ABOUT THE DATASET**

### Dataset inspection

This subsection provides a preliminary overview of the dataset structure.  
It checks the number of rows and columns, the available metadata fields, and the distribution of speeches by year, month, and location.

In [ ]:
df.info()